# Data Cleaning Assignment 2 — 20 Prompts and Results

**Dataset:** Titanic passenger dataset, downloaded from
[datasciencedojo/datasets](https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv)
(`titanic.csv`, 891 rows x 12 columns).

Each numbered prompt below states a data-cleaning task; the code cell
immediately under it performs the task on a *live* DataFrame (state
carries over from one prompt to the next) and the printed output shows
the real before/after result.

### Prompt 1: Load the dataset and inspect its basic structure
Load `titanic.csv` (downloaded from
https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv)
into a pandas DataFrame and display its shape, column names, data types and
the first few rows.

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

df = pd.read_csv('titanic.csv')
original_df = df.copy()  # keep an untouched copy for before/after comparison

print("Shape (rows, columns):", df.shape)
print("\nColumn names:", list(df.columns))
print("\nData types:\n", df.dtypes)
print("\nFirst 5 rows:\n", df.head())

Shape (rows, columns): (891, 12)

Column names: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']

Data types:
 PassengerId      int64
Survived         int64
Pclass           int64
Name               str
Sex                str
Age            float64
SibSp            int64
Parch            int64
Ticket             str
Fare           float64
Cabin              str
Embarked           str
dtype: object

First 5 rows:
    PassengerId  Survived  Pclass                                               Name     Sex   Age  SibSp  Parch  \
0            1         0       3                            Braund, Mr. Owen Harris    male  22.0      1      0   
1            2         1       1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1      0   
2            3         1       3                             Heikkinen, Miss. Laina  female  26.0      0      0   
3            4         1       1       Futrelle, Mrs. J

### Prompt 2: Get an overall summary / profile of the dataset
Use `.info()` and `.describe()` to understand numeric ranges, counts and
possible data quality issues before cleaning anything.

In [ ]:
print(df.info())
print("\nNumeric summary:\n", df.describe())
print("\nCategorical summary:\n", df.describe(include=['object']))

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB
None

Numeric summary:
        PassengerId    Survived      Pclass         Age       SibSp       Parch        Fare
count   891.000000  891.000000  891.000000  714.000000  891.000000  891.000000  891.000000
mean    446.000000    0.383838    2.308642   29.69911

### Prompt 3: Detect missing values in every column
Compute the count and percentage of missing values per column and identify
which columns need attention.

In [ ]:
missing_count = df.isnull().sum()
missing_pct = (missing_count / len(df) * 100).round(2)
missing_report = pd.DataFrame({'missing_count': missing_count,
                                'missing_pct': missing_pct})
missing_report = missing_report[missing_report['missing_count'] > 0] \
                    .sort_values('missing_count', ascending=False)
print(missing_report)

          missing_count  missing_pct
Cabin               687        77.10
Age                 177        19.87
Embarked              2         0.22


### Prompt 4: Impute missing 'Age' values intelligently
Instead of a single global mean, fill missing `Age` using the median age of
each `Pclass`/`Sex` group, which is more representative than one overall
statistic.

In [ ]:
print("Missing Age before:", df['Age'].isnull().sum())

df['Age'] = df.groupby(['Pclass', 'Sex'])['Age'] \
              .transform(lambda s: s.fillna(s.median()))

print("Missing Age after:", df['Age'].isnull().sum())
print("\nAge summary after imputation:\n", df['Age'].describe())

Missing Age before: 177
Missing Age after: 0

Age summary after imputation:
 count    891.000000
mean      29.112424
std       13.304424
min        0.420000
25%       21.500000
50%       26.000000
75%       36.000000
max       80.000000
Name: Age, dtype: float64


### Prompt 5: Impute missing 'Embarked' values using the mode
`Embarked` only has a couple of missing values, so fill them with the most
frequent port of embarkation.

In [ ]:
print("Missing Embarked before:", df['Embarked'].isnull().sum())
print("Value counts before:\n", df['Embarked'].value_counts(dropna=False))

mode_embarked = df['Embarked'].mode()[0]
df['Embarked'] = df['Embarked'].fillna(mode_embarked)

print("\nFilled missing Embarked with mode:", mode_embarked)
print("Missing Embarked after:", df['Embarked'].isnull().sum())

Missing Embarked before: 2
Value counts before:
 Embarked
S      644
C      168
Q       77
NaN      2
Name: count, dtype: int64

Filled missing Embarked with mode: S
Missing Embarked after: 0


### Prompt 6: Handle the heavily-missing 'Cabin' column
`Cabin` is missing for ~77% of passengers, so instead of imputing a fake
value, engineer a useful binary feature `Has_Cabin` and a `Deck` feature
(first letter of the cabin code, 'Unknown' if missing).

In [ ]:
print("Missing Cabin before:", df['Cabin'].isnull().sum(),
      f"({df['Cabin'].isnull().mean()*100:.2f}%)")

df['Has_Cabin'] = df['Cabin'].notnull().astype(int)
df['Deck'] = df['Cabin'].str[0].fillna('Unknown')

print("\nHas_Cabin value counts:\n", df['Has_Cabin'].value_counts())
print("\nDeck value counts:\n", df['Deck'].value_counts())

Missing Cabin before: 687 (77.10%)

Has_Cabin value counts:
 Has_Cabin
0    687
1    204
Name: count, dtype: int64

Deck value counts:
 Deck
Unknown    687
C           59
B           47
D           33
E           32
A           15
F           13
G            4
T            1
Name: count, dtype: int64


### Prompt 7: Detect and remove duplicate rows
Check whether the dataset contains fully duplicated rows or duplicated
passenger identifiers, and remove them.

In [ ]:
full_dupes = df.duplicated().sum()
id_dupes = df['PassengerId'].duplicated().sum()
print("Fully duplicated rows:", full_dupes)
print("Duplicated PassengerId values:", id_dupes)

before_rows = len(df)
df = df.drop_duplicates()
print(f"\nRows before: {before_rows}, rows after drop_duplicates: {len(df)}")

Fully duplicated rows: 0
Duplicated PassengerId values: 0

Rows before: 891, rows after drop_duplicates: 891


### Prompt 8: Standardize inconsistent text formatting
Strip stray whitespace and normalize the case of text/categorical columns
(`Name`, `Sex`, `Embarked`) so values compare and group correctly.

In [ ]:
print("Sex categories before:", df['Sex'].unique())
print("Embarked categories before:", df['Embarked'].unique())

df['Name'] = df['Name'].str.strip()
df['Sex'] = df['Sex'].str.strip().str.lower()
df['Embarked'] = df['Embarked'].str.strip().str.upper()

print("\nSex categories after:", df['Sex'].unique())
print("Embarked categories after:", df['Embarked'].unique())

Sex categories before: <StringArray>
['male', 'female']
Length: 2, dtype: str
Embarked categories before: <StringArray>
['S', 'C', 'Q']
Length: 3, dtype: str

Sex categories after: <StringArray>
['male', 'female']
Length: 2, dtype: str
Embarked categories after: <StringArray>
['S', 'C', 'Q']
Length: 3, dtype: str


### Prompt 9: Extract a 'Title' feature from the 'Name' text field
Parse each passenger's honorific title (Mr, Mrs, Miss, etc.) out of the
free-text `Name` column and collapse rare titles into an 'Other' bucket.

In [ ]:
df['Title'] = df['Name'].str.extract(r',\s*([^.]*)\.')[0].str.strip()
print("Raw title counts:\n", df['Title'].value_counts())

common_titles = {'Mr', 'Mrs', 'Miss', 'Master'}
df['Title'] = df['Title'].apply(lambda t: t if t in common_titles else 'Other')

print("\nCleaned title counts:\n", df['Title'].value_counts())

Raw title counts:
 Title
Mr              517
Miss            182
Mrs             125
Master           40
Dr                7
Rev               6
Major             2
Mlle              2
Col               2
Don               1
Mme               1
Ms                1
Lady              1
Sir               1
Capt              1
the Countess      1
Jonkheer          1
Name: count, dtype: int64

Cleaned title counts:
 Title
Mr        517
Miss      182
Mrs       125
Master     40
Other      27
Name: count, dtype: int64


### Prompt 10: Convert columns to appropriate data types
`Survived`, `Pclass`, `Sex`, `Embarked`, `Title` and `Deck` are categorical
in nature even though some are stored as integers/strings — convert them to
pandas `category` dtype to save memory and make intent explicit.

In [ ]:
print("Memory usage before:\n", df.memory_usage(deep=True).sum(), "bytes")

cat_cols = ['Survived', 'Pclass', 'Sex', 'Embarked', 'Title', 'Deck']
for col in cat_cols:
    df[col] = df[col].astype('category')

print("\nDtypes after conversion:\n", df.dtypes)
print("\nMemory usage after:", df.memory_usage(deep=True).sum(), "bytes")

Memory usage before:
 438869 bytes

Dtypes after conversion:
 PassengerId       int64
Survived       category
Pclass         category
Name                str
Sex            category
Age             float64
SibSp             int64
Parch             int64
Ticket              str
Fare            float64
Cabin               str
Embarked       category
Has_Cabin         int64
Deck           category
Title          category
dtype: object

Memory usage after: 215374 bytes


### Prompt 11: Detect outliers in 'Fare' using the IQR method
Compute Q1, Q3 and the interquartile range to flag fares that fall far
outside the typical range.

In [ ]:
Q1 = df['Fare'].quantile(0.25)
Q3 = df['Fare'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[(df['Fare'] < lower_bound) | (df['Fare'] > upper_bound)]
print(f"Q1={Q1:.2f}, Q3={Q3:.2f}, IQR={IQR:.2f}")
print(f"Valid range: [{lower_bound:.2f}, {upper_bound:.2f}]")
print(f"Number of Fare outliers: {len(outliers)} ({len(outliers)/len(df)*100:.2f}%)")
print("\nSample outlier fares:\n", outliers['Fare'].sort_values(ascending=False).head())

Q1=7.91, Q3=31.00, IQR=23.09
Valid range: [-26.72, 65.63]
Number of Fare outliers: 116 (13.02%)

Sample outlier fares:
 258    512.3292
737    512.3292
679    512.3292
27     263.0000
341    263.0000
Name: Fare, dtype: float64


### Prompt 12: Treat 'Fare' outliers by capping (winsorizing)
Rather than deleting outlier rows and losing data, cap extreme fares at the
IQR bounds computed in the previous step.

In [ ]:
print("Fare stats before capping:\n", df['Fare'].describe())

df['Fare'] = df['Fare'].clip(lower=lower_bound, upper=upper_bound)

print("\nFare stats after capping:\n", df['Fare'].describe())

Fare stats before capping:
 count    891.000000
mean      32.204208
std       49.693429
min        0.000000
25%        7.910400
50%       14.454200
75%       31.000000
max      512.329200
Name: Fare, dtype: float64

Fare stats after capping:
 count    891.000000
mean      24.046813
std       20.481625
min        0.000000
25%        7.910400
50%       14.454200
75%       31.000000
max       65.634400
Name: Fare, dtype: float64


### Prompt 13: Detect outliers in 'Age' using the Z-score method
Use standard scores to flag passengers whose age is more than 3 standard
deviations from the mean.

In [ ]:
z_scores = (df['Age'] - df['Age'].mean()) / df['Age'].std()
age_outliers = df[z_scores.abs() > 3]

print(f"Mean age: {df['Age'].mean():.2f}, Std: {df['Age'].std():.2f}")
print(f"Number of Age outliers (|z|>3): {len(age_outliers)}")
print("\nOutlier ages:\n", age_outliers['Age'].sort_values(ascending=False))

Mean age: 29.11, Std: 13.30
Number of Age outliers (|z|>3): 7

Outlier ages:
 630    80.0
851    74.0
96     71.0
493    71.0
116    70.5
672    70.0
745    70.0
Name: Age, dtype: float64


### Prompt 14: Validate logical/range constraints
Confirm that `Age`, `Fare`, `SibSp` and `Parch` never hold impossible
negative values, and flag/fix any row that violates the constraint.

In [ ]:
constraints = {
    'Age': df['Age'] < 0,
    'Fare': df['Fare'] < 0,
    'SibSp': df['SibSp'] < 0,
    'Parch': df['Parch'] < 0,
}
for col, mask in constraints.items():
    n_invalid = mask.sum()
    print(f"{col}: {n_invalid} invalid (negative) values")
    if n_invalid:
        df.loc[mask, col] = np.nan

print("\nAll range constraints satisfied:",
      all((df[c] >= 0).all() for c in constraints))

Age: 0 invalid (negative) values
Fare: 0 invalid (negative) values
SibSp: 0 invalid (negative) values
Parch: 0 invalid (negative) values

All range constraints satisfied: True


### Prompt 15: Engineer 'FamilySize' and 'IsAlone' features
Combine `SibSp` and `Parch` into a single family-size measure and derive a
boolean flag for passengers travelling alone — useful, analysis-ready
features that don't exist in the raw data.

In [ ]:
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

print(df[['SibSp', 'Parch', 'FamilySize', 'IsAlone']].head(10))
print("\nIsAlone distribution:\n", df['IsAlone'].value_counts())

   SibSp  Parch  FamilySize  IsAlone
0      1      0           2        0
1      1      0           2        0
2      0      0           1        1
3      1      0           2        0
4      0      0           1        1
5      0      0           1        1
6      0      0           1        1
7      3      1           5        0
8      0      2           3        0
9      1      0           2        0

IsAlone distribution:
 IsAlone
1    537
0    354
Name: count, dtype: int64


### Prompt 16: Encode categorical variables for modeling
Apply label encoding to the binary `Sex` column and one-hot encoding to the
multi-class `Embarked` column so the dataset is ready for machine-learning
use.

In [ ]:
df['Sex_encoded'] = df['Sex'].map({'male': 0, 'female': 1})

embarked_dummies = pd.get_dummies(df['Embarked'], prefix='Embarked')
df = pd.concat([df, embarked_dummies], axis=1)

print(df[['Sex', 'Sex_encoded']].drop_duplicates())
print("\nOne-hot encoded Embarked columns:\n",
      df[list(embarked_dummies.columns)].head())

      Sex Sex_encoded
0    male           0
1  female           1

One-hot encoded Embarked columns:
    Embarked_C  Embarked_Q  Embarked_S
0       False       False        True
1        True       False       False
2       False       False        True
3       False       False        True
4       False       False        True


### Prompt 17: Drop irrelevant / redundant columns
Remove identifier and free-text columns (`PassengerId`, `Ticket`, `Name`,
`Cabin`) that carried no direct analytical value once their useful
information (Title, Deck, Has_Cabin) has already been extracted.

In [ ]:
cols_to_drop = ['PassengerId', 'Ticket', 'Name', 'Cabin']
print("Columns before drop:", list(df.columns))

df = df.drop(columns=cols_to_drop)

print("\nColumns after drop:", list(df.columns))
print("New shape:", df.shape)

Columns before drop: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked', 'Has_Cabin', 'Deck', 'Title', 'FamilySize', 'IsAlone', 'Sex_encoded', 'Embarked_C', 'Embarked_Q', 'Embarked_S']

Columns after drop: ['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked', 'Has_Cabin', 'Deck', 'Title', 'FamilySize', 'IsAlone', 'Sex_encoded', 'Embarked_C', 'Embarked_Q', 'Embarked_S']
New shape: (891, 17)


### Prompt 18: Reduce skewness in 'Fare' with a log transform
Right-skewed monetary values like `Fare` distort statistics and models —
compare skewness before and after applying `log1p` and keep the
transformed version as a new column.

In [ ]:
skew_before = df['Fare'].skew()
df['Fare_log'] = np.log1p(df['Fare'])
skew_after = df['Fare_log'].skew()

print(f"Skewness of Fare before log transform: {skew_before:.3f}")
print(f"Skewness of Fare after log1p transform: {skew_after:.3f}")
print("\n", df[['Fare', 'Fare_log']].describe())

Skewness of Fare before log transform: 1.082
Skewness of Fare after log1p transform: -0.238

              Fare    Fare_log
count  891.000000  891.000000
mean    24.046813    2.893539
std     20.481625    0.835804
min      0.000000    0.000000
25%      7.910400    2.187218
50%     14.454200    2.737881
75%     31.000000    3.465736
max     65.634400    4.199221


### Prompt 19: Rename columns to a consistent snake_case convention
Standardize column naming across the whole DataFrame so downstream code and
SQL exports don't have to juggle mixed CamelCase / PascalCase names.

In [ ]:
import re

def to_snake_case(name):
    name = re.sub(r'(?<!^)(?=[A-Z])', '_', name)
    return name.lower()

print("Columns before renaming:\n", list(df.columns))

df.columns = [to_snake_case(c) for c in df.columns]

print("\nColumns after renaming:\n", list(df.columns))

Columns before renaming:
 ['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked', 'Has_Cabin', 'Deck', 'Title', 'FamilySize', 'IsAlone', 'Sex_encoded', 'Embarked_C', 'Embarked_Q', 'Embarked_S', 'Fare_log']

Columns after renaming:
 ['survived', 'pclass', 'sex', 'age', 'sib_sp', 'parch', 'fare', 'embarked', 'has__cabin', 'deck', 'title', 'family_size', 'is_alone', 'sex_encoded', 'embarked__c', 'embarked__q', 'embarked__s', 'fare_log']


### Prompt 20: Final validation and export the cleaned dataset
Run a last missing-value / duplicate check, compare the cleaned dataset
against the original raw file, and save the result to
`titanic_cleaned.csv`.

In [ ]:
print("=== FINAL VALIDATION ===")
print("Remaining missing values:\n", df.isnull().sum()[df.isnull().sum() > 0])

feature_dupes = df.duplicated().sum()
print(f"\nRows sharing identical values across all remaining feature columns: {feature_dupes}")
print("These are NOT re-run duplicates of Prompt 7 (which found 0 exact-row")
print("duplicates on the raw data). They appear only now because Prompt 17")
print("dropped the identifying columns (PassengerId, Name, Ticket, Cabin) -")
print("distinct real passengers can legitimately share the same Pclass/Sex/")
print("Age/Fare/etc. profile, so these rows are kept rather than dropped.")

print("\n=== BEFORE vs AFTER SUMMARY ===")
print(f"Original shape : {original_df.shape}")
print(f"Cleaned shape  : {df.shape}")
print(f"Original missing values (total): {original_df.isnull().sum().sum()}")
print(f"Cleaned missing values (total) : {df.isnull().sum().sum()}")
print(f"Original memory usage: {original_df.memory_usage(deep=True).sum()} bytes")
print(f"Cleaned memory usage : {df.memory_usage(deep=True).sum()} bytes")

df.to_csv('titanic_cleaned.csv', index=False)
print("\nSaved cleaned dataset to 'titanic_cleaned.csv'")

=== FINAL VALIDATION ===
Remaining missing values:
 Series([], dtype: int64)

Rows sharing identical values across all remaining feature columns: 110
These are NOT re-run duplicates of Prompt 7 (which found 0 exact-row
duplicates on the raw data). They appear only now because Prompt 17
dropped the identifying columns (PassengerId, Name, Ticket, Cabin) -
distinct real passengers can legitimately share the same Pclass/Sex/
Age/Fare/etc. profile, so these rows are kept rather than dropped.

=== BEFORE vs AFTER SUMMARY ===
Original shape : (891, 12)
Cleaned shape  : (891, 18)
Original missing values (total): 866
Cleaned missing values (total) : 0
Original memory usage: 322592 bytes
Cleaned memory usage : 67253 bytes

Saved cleaned dataset to 'titanic_cleaned.csv'
